# MOHIM stem-wise motif threshold diagnostics

`songs` 음원을 source 단위로 분리하고, 앞 30초의 모든 4마디 후보 점수를 threshold 적용 없이 확인합니다.

## 0. Drive와 motif 브랜치 준비

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import subprocess

REPOSITORY = 'https://github.com/youhan200203/MOHIM.git'
BRANCH = 'motif'
REPO_DIR = Path('/content/MOHIM')
if not (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', 'clone', '-b', BRANCH, REPOSITORY, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'switch', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
print('repository:', REPO_DIR)

In [ ]:
%pip install -q -r requirements.txt
%pip install -q --no-deps "beat-this @ git+https://github.com/CPJKU/beat_this.git"

## 1. 실험 경로와 범위

In [ ]:
MAX_SONGS = 10
DEVICE = 'cuda'
MOTIF_BARS = 4
MOTIF_SEARCH_SECONDS = 30.0

SONGS_DIR = Path('/content/drive/MyDrive/MOHIM/songs')
DIAGNOSTIC_DIR = Path('/content/drive/MyDrive/MOHIM/motif_stem_diagnostics')
BEAT_CHECKPOINT = Path('/content/checkpoints/beat_this_final0.ckpt')
AUDIO_EXTENSIONS = {'.aac', '.flac', '.m4a', '.mp3', '.ogg', '.wav', '.webm'}

song_paths = sorted(
    path for path in SONGS_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS
)[:MAX_SONGS]
assert song_paths, f'음원 파일이 없습니다: {SONGS_DIR}'
DIAGNOSTIC_DIR.mkdir(parents=True, exist_ok=True)
print('songs:', len(song_paths))
for path in song_paths:
    print('-', path.name)

## 2. Demucs와 motif scorer 준비

In [ ]:
import urllib.request
from mohim.motif import MotifConfig, MotifExtractor, create_beat_tracker
from mohim.separator import StemSeparator

BEAT_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
if not BEAT_CHECKPOINT.is_file():
    urllib.request.urlretrieve(
        'https://cloud.cp.jku.at/public.php/dav/files/7ik4RrBKTS273gp/final0.ckpt', BEAT_CHECKPOINT
    )
separator = StemSeparator(device=DEVICE, model_name='htdemucs_6s')
beat_tracker = create_beat_tracker(BEAT_CHECKPOINT, device=DEVICE)
motif_scorer = MotifExtractor(
    beat_tracker,
    MotifConfig(bars=MOTIF_BARS, search_seconds=MOTIF_SEARCH_SECONDS),
)
print('separator and motif scorer ready')

## 3. 모든 stem × start-downbeat 후보 계산

`active_ratio >= 0.8`이고 onset/chroma similarity 차이가 0.4 이하인 모든 후보를 저장합니다.

In [ ]:
import json
import pandas as pd
from mohim.separator import save_audio

all_rows = []
for song_index, audio_path in enumerate(song_paths):
    print(f'[{song_index + 1}/{len(song_paths)}] {audio_path.name}')
    song_id = f'{song_index:03d}_{audio_path.stem}'
    song_dir = DIAGNOSTIC_DIR / song_id
    song_dir.mkdir(parents=True, exist_ok=True)
    stems, sample_rate, _ = separator.separate(audio_path)
    result = motif_scorer.score_all(audio_path, stems, sample_rate)
    melodic = result['melodic_accompaniment']
    save_audio(song_dir / 'melodic_accompaniment.flac', melodic, sample_rate, audio_format='flac')

    for row_index, row in enumerate(result['candidates']):
        start = round(row['start_sec'] * sample_rate)
        end = round(row['end_sec'] * sample_rate)
        filename = f"candidate_{row_index:03d}_{row['stem_name']}.flac"
        save_audio(song_dir / filename, melodic[:, start:end], sample_rate, audio_format='flac')
        row['candidate_file'] = filename
        all_rows.append({'song_id': song_id, 'title': audio_path.stem, **row})

    metadata = {
        'song_id': song_id, 'title': audio_path.stem, 'source_audio': str(audio_path),
        'sample_rate': sample_rate, 'candidates': result['candidates'],
    }
    (song_dir / 'motif_scores.json').write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    del stems, result, melodic

scores_df = pd.DataFrame(all_rows)
scores_df.to_csv(DIAGNOSTIC_DIR / 'all_motif_scores.csv', index=False)
display(scores_df.sort_values(['song_id', 'start_sec', 'stem_name']))

## 4. 곡과 후보를 골라 듣기

점수표의 행을 확인한 뒤 `DEBUG_TRACK_INDEX`, `DEBUG_CANDIDATE_INDEX`를 바꿉니다. 재생되는 오디오는 검출 stem 단독이 아니라 같은 시간대의 drums 제외 accompaniment입니다.

In [ ]:
from IPython.display import Audio, display

DEBUG_TRACK_INDEX = 0
DEBUG_CANDIDATE_INDEX = 0
metadata_paths = sorted(DIAGNOSTIC_DIR.glob('*/motif_scores.json'))
assert metadata_paths, '완료된 진단 결과가 없습니다.'
metadata_path = metadata_paths[DEBUG_TRACK_INDEX]
metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
candidate_df = pd.DataFrame(metadata['candidates'])
display(candidate_df.sort_values(['start_sec', 'stem_name']))
row = metadata['candidates'][DEBUG_CANDIDATE_INDEX]
print({key: value for key, value in row.items() if key != 'candidate_file'})
display(Audio(filename=str(metadata_path.parent / row['candidate_file'])))